# 🧠 F1 Score และการปรับจูนค่า F-Beta

ยินดีต้อนรับสู่โน้ตบุ๊กอธิบายภาคปฏิบัติเกี่ยวกับ **F1 Score** และ **F-Beta Score**! ในโน้ตบุ๊กนี้ เราจะ:
1. กำหนดสูตรทางคณิตศาสตร์สำหรับ F1 Score และ F-Beta Score ทั่วไป
2. เขียนฟังก์ชันคำนวณ F1 และ F-Beta จากศูนย์ (from scratch) โดยใช้ NumPy และตรวจสอบความถูกต้องเทียบกับ `scikit-learn`
3. จำลองผลลัพธ์การตรวจจับ bounding box เพื่อคำนวณ Precision, Recall, ค่าเฉลี่ยเลขคณิต (Arithmetic Mean), F1 และ F-Beta ในช่วงเกณฑ์ความมั่นใจ (confidence threshold) ต่าง ๆ
4. แสดงภาพกราฟ **F1-Threshold Curve** (เช่นเดียวกับไฟล์ `BoxF1_curve.png` ของ YOLO) และระบุตำแหน่งเกณฑ์ที่เหมาะสมที่สุด (optimal threshold) ในการรักษาสมดุลระหว่าง precision และ recall
5. พล็อตและเปรียบเทียบกราฟของ **F0.5** (ที่เน้น Precision) และ **F2** (ที่เน้น Recall) เพื่อดูการเปลี่ยนแปลงของเกณฑ์ที่เหมาะสมที่สุดตามความต้องการทางธุรกิจ

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลย

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import f1_score, fbeta_score

# Set seed for reproducibility
np.random.seed(42)

## 1. การคำนวณ F1 และ F-Beta จากศูนย์ (from Scratch)

ลองเขียนฟังก์ชันสำหรับคำนวณตัววัดประสิทธิภาพทั้งสองตัวกัน:
-   F1 Score มาตรฐาน:
    $$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
-   F-Beta Score ทั่วไป:
    $$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{(\beta^2 \cdot \text{Precision}) + \text{Recall}}$$

In [ ]:
def custom_fbeta(y_true, y_pred, beta=1.0):
    """
    Calculate the F-Beta score from scratch.
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    
    if precision == 0.0 or recall == 0.0:
        return 0.0
        
    num = (1 + beta**2) * precision * recall
    den = (beta**2 * precision) + recall
    return num / den

def custom_f1(y_true, y_pred):
    return custom_fbeta(y_true, y_pred, beta=1.0)

# Test arrays
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

f1_scratch = custom_f1(y_true, y_pred)
f1_sklearn = f1_score(y_true, y_pred)
f2_scratch = custom_fbeta(y_true, y_pred, beta=2.0)
f2_sklearn = fbeta_score(y_true, y_pred, beta=2.0)

print(f"Custom F1       : {f1_scratch:.4f} | Sklearn F1       : {f1_sklearn:.4f}")
print(f"Custom F2 (Beta2): {f2_scratch:.4f} | Sklearn F2 (Beta2): {f2_sklearn:.4f}")

## 2. การกวาดหาค่า Threshold และการพล็อต F1 Curve

มาจำลองข้อมูล bounding box ขึ้นมาใหม่ตามตัวอย่างใน EX13 และ EX14:
-   wellhead จริง 30 ชิ้น (Class 1) ที่มีค่าความมั่นใจ (confidence score) สูง
-   พื้นที่พื้นหลัง (background) จริง 170 บริเวณ (Class 0) ที่มีค่าความมั่นใจต่ำ

ลองคำนวณค่า F1, F0.5 และ F2 ในช่วงเกณฑ์ความมั่นใจตั้งแต่ 0.0 ถึง 0.95

In [ ]:
# Generate 200 samples
n_samples = 200
y_true_wellheads = np.concatenate([np.ones(30), np.zeros(170)]).astype(int)

# Generate confidence scores
conf_wellheads = np.random.normal(0.8, 0.15, 30)
conf_bg = np.random.normal(0.3, 0.18, 170)
confidence_scores = np.concatenate([conf_wellheads, conf_bg])
confidence_scores = np.clip(confidence_scores, 0.0, 1.0)

ตอนนี้มาทำการกวาดหาค่า thresholds และติดตามตัววัดประสิทธิภาพต่าง ๆ

In [ ]:
thresholds = np.linspace(0.0, 0.95, 100)
f1_history = []
f05_history = []
f2_history = []

for threshold in thresholds:
    y_pred_temp = (confidence_scores >= threshold).astype(int)
    
    f1_val = custom_fbeta(y_true_wellheads, y_pred_temp, beta=1.0)
    f05_val = custom_fbeta(y_true_wellheads, y_pred_temp, beta=0.5)
    f2_val = custom_fbeta(y_true_wellheads, y_pred_temp, beta=2.0)
    
    f1_history.append(f1_val)
    f05_history.append(f05_val)
    f2_history.append(f2_val)

# Find optimal thresholds
opt_idx_f1 = np.argmax(f1_history)
opt_thresh_f1 = thresholds[opt_idx_f1]
max_f1 = f1_history[opt_idx_f1]

# Plot F1 Curve vs Confidence Threshold
plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_history, color='dodgerblue', linewidth=3, label=f'F1 (Best: {max_f1:.2f} at Thresh={opt_thresh_f1:.2f})')
plt.xlabel('Confidence Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs. Confidence Threshold (YOLO F1-Curve Style)')
plt.axvline(opt_thresh_f1, color='red', linestyle='--', alpha=0.8, label=f'Optimal Threshold ({opt_thresh_f1:.2f})')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

นี่คือวิธีที่โมเดล YOLO ใช้คำนวณเกณฑ์ความมั่นใจที่เหมาะสมที่สุด (optimal confidence threshold) ระหว่างการทำ validation! ในการจำลองนี้ การตั้งค่า `conf=0.61` จะให้ค่า F1 score ที่สมดุลและสูงที่สุด

## 3. การปรับแต่งความสมดุลของการแลกเปลี่ยน (การเปรียบเทียบ F-Beta)

มาเปรียบเทียบเส้นโค้ง F0.5 และ F2 เพื่อดูว่าเกณฑ์ที่เหมาะสมที่สุดจะเปลี่ยนแปลงอย่างไร

In [ ]:
opt_thresh_f05 = thresholds[np.argmax(f05_history)]
opt_thresh_f2 = thresholds[np.argmax(f2_history)]

plt.figure(figsize=(12, 6))
plt.plot(thresholds, f1_history, color='dodgerblue', linewidth=2.5, label='F1 (Equal weight)')
plt.plot(thresholds, f05_history, color='green', linewidth=2.5, linestyle='-.', label='F0.5 (Prioritizes Precision)')
plt.plot(thresholds, f2_history, color='orange', linewidth=2.5, linestyle='--', label='F2 (Prioritizes Recall)')

plt.axvline(opt_thresh_f05, color='green', linestyle=':', alpha=0.8, label=f'F0.5 Opt: {opt_thresh_f05:.2f}')
plt.axvline(opt_thresh_f1, color='dodgerblue', linestyle=':', alpha=0.8, label=f'F1 Opt: {opt_thresh_f1:.2f}')
plt.axvline(opt_thresh_f2, color='orange', linestyle=':', alpha=0.8, label=f'F2 Opt: {opt_thresh_f2:.2f}')

plt.xlabel('Confidence Threshold')
plt.ylabel('Score')
plt.title('F-Beta Score Comparison: How Beta Shifts the Optimal Threshold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

ข้อสังเกต:
-   **F0.5 (เน้น Precision):** เกณฑ์ที่เหมาะสมที่สุดจะเลื่อน **สูงขึ้น** ($\approx 0.72$) เพื่อกรองการแจ้งเตือนที่ผิดพลาด (false alarms) ออกไปให้มากที่สุด
-   **F2 (เน้น Recall):** เกณฑ์ที่เหมาะสมที่สุดจะเลื่อน **ต่ำลง** ($\approx 0.44$) เพื่อให้ครอบคลุมการตรวจพบ wellhead จริง ๆ ให้ได้มากที่สุด

## 💡 ความเชื่อมโยงกับ Computer Vision และ YOLO
*   **เกณฑ์ที่เหมาะสมที่สุดสำหรับการนำไปใช้งานจริง (Optimal Deployment Threshold):** เมื่อคุณทำการทดสอบประเมินผล (validate) โมเดล YOLO ที่ฝึกสอนเสร็จแล้ว ระบบจะรายงานผลค่า precision, recall และ mAP50 และหากต้องการหาเกณฑ์ที่ให้ค่า F1 score ที่ดีที่สุด คุณสามารถตรวจสอบได้จากกราฟ `BoxF1_curve.png` ในโฟลเดอร์ผลการฝึกสอน เมื่อคุณจะนำโมเดลไปใช้งานจริง (deployment) คุณสามารถกำหนดค่าเกณฑ์ที่เหมาะสมที่สุดนี้ได้โดยระบุที่พารามิเตอร์ `conf`